# Nexto Distillation — Dataset Generation & BC Training

This notebook generates a behavior-cloning dataset from the Nexto teacher bot,
then trains a student MLP to imitate it.

**Runtime:** Go to **Runtime → Change runtime type → T4 GPU** before starting.

## 1. Clone Repo

In [ ]:
!git clone https://github.com/AdamTheGans/Rocket-League-Bot.git
%cd Rocket-League-Bot

## 2. Install Dependencies

Colab already has PyTorch + CUDA. We just need rlgym + RocketSim + extras.

In [ ]:
# RLGym v2 + RocketSim (the physics simulator)
!pip install rlgym[rl-sim]

# PPO engine (needed for some imports)
!pip install git+https://github.com/AechPro/rlgym-ppo

# Other deps from requirements.txt
!pip install numpy==1.26.4 cloudpickle==3.1.2

## 3. Verify Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import RocketSim
print(f"RocketSim: OK")

import rlgym
print(f"RLGym: OK")

import os
assert os.path.isfile("nexto/nexto-model.pt"), "Nexto model not found!"
print(f"Nexto model: Found")

print("\n=== ALL CHECKS PASSED ===")

## 4. Generate Dataset

This rolls out the Nexto teacher in RocketSim and collects
`(student_obs, teacher_action, teacher_logits)` data.

- **1M steps** ≈ 30-80 min depending on GPU/CPU
- Uses CUDA for Nexto inference, RocketSim runs on CPU (always)
- Shards saved to `data/nexto_distill/shards/`

In [ ]:
%cd src

!python -m nexto_distill.generate_dataset \
    --out_dir ../data/nexto_distill/shards \
    --num_steps 1000000 \
    --shard_size 50000 \
    --seed 42 \
    --device cuda \
    --report_every 10000

## 5. Train Student (Behavior Cloning)

Trains the student MLP to imitate Nexto's actions.
This is pure PyTorch — runs fast on GPU.

In [ ]:
!python -m nexto_distill.train_bc \
    --data_dir ../data/nexto_distill/shards \
    --layers 2048 1024 1024 512 \
    --lr 3e-4 \
    --epochs 50 \
    --batch_size 4096 \
    --checkpoint_dir ../checkpoints/nexto_distill \
    --device cuda \
    --seed 42

## 6. Evaluate (Offline)

In [ ]:
!python -m nexto_distill.eval_imitation \
    --checkpoint ../checkpoints/nexto_distill/student_policy.pt \
    --mode offline \
    --data_dir ../data/nexto_distill/shards \
    --device cuda

## 7. Evaluate (Online — Student plays in RocketSim)

In [ ]:
!python -m nexto_distill.eval_imitation \
    --checkpoint ../checkpoints/nexto_distill/student_policy.pt \
    --mode online \
    --episodes 50 \
    --device cpu

## 8. Download Results

Download the trained student policy and dataset to your local machine.

In [ ]:
# Zip up the checkpoint
!zip -r /content/nexto_distill_results.zip \
    ../checkpoints/nexto_distill/ \
    ../data/nexto_distill/shards/metadata.json

from google.colab import files
files.download('/content/nexto_distill_results.zip')